# Lab: Support Vector Machines
*In this lab, we will apply Support Vector Machine (SVM) techniques using scikit-learn, guided by the [Foundational Methodology for Data Science](../../../05_methodology/). While real-world projects require comprehensive documentation at each stage, this lab focuses on a streamlined, practical approach to demonstrate the core concepts and workflow. Therefore we will settle with the summaries of hypothetical stage reports.*

---

> ### 📝 1. Business Understanding Report (Summary)
>
> * **Business Context:** Credit card fraud is a critical problem for financial institutions and payment processors, resulting in significant financial losses and eroding consumer confidence. As digital transactions become more common, the volume and sophistication of fraudulent activity continue to increase, challenging traditional detection methods.
> * **Business Problem:** The current approach to identifying fraudulent transactions is largely reactive and relies on static rules or manual review. This results in delayed detection, increased operational costs, and dissatisfied customers who experience either false alarms or undetected fraudulent charges on their accounts.
> * **Project Goal:** The objective of this project is to enable the early and accurate identification of fraudulent credit card transactions within a large volume of real-time payments. By moving from manual processes and fixed rules to an automated, data-driven system, the business aims to minimize financial losses, improve operational efficiency, and enhance the customer experience.
> * **Business Impact:** A successful solution would allow the company to:
>   - Prevent significant monetary losses caused by undetected fraudulent activity.
>   - Respond rapidly to suspicious transactions and deploy targeted interventions.
>   - Reduce false positives, minimizing unnecessary inconvenience for legitimate customers.
>   - Strengthen trust and brand reputation in a highly competitive financial market.

---

> ### 📝 2. Analytic Approach Report (Summary)
>
> * **Problem Type:** This is a classic **supervised binary classification problem**, where each credit card transaction must be classified as either “fraudulent” or “legitimate.” The dataset is highly imbalanced, with fraudulent transactions making up only a small fraction of total records.
> * **Modeling Approach:** To address the business objective, we will develop and evaluate a machine learning model that can automate the identification of fraudulent transactions. The main focus will be on applying **Support Vector Machines (SVM)**, a powerful family of classification algorithms well-suited for separating complex, high-dimensional data, especially after proper feature scaling.
> * **Handling Class Imbalance:** Due to the extreme imbalance in the dataset, special care will be taken to ensure the model does not default to predicting the majority class (legitimate transactions). Strategies may include:
>   - Using appropriate evaluation metrics that reflect performance on the minority (fraud) class.
>   - Exploring techniques such as data resampling, class weighting, or anomaly detection methods where relevant.
> * **Evaluation Metrics:** Given the cost of false negatives (missed fraud) and false positives (legitimate transactions flagged), multiple metrics will be applied:
>   - **Recall (Sensitivity):** Primary metric, maximizing the detection rate of fraudulent transactions.
>   - **Precision:** Identifying how many flagged cases are actually fraud.
>   - **F1-Score:** Balance between precision and recall.
>   - **ROC-AUC:** Performance across thresholds.
> Overall accuracy will **not** be the main metric due to class imbalance.
> * **Validation Strategy:** The dataset will be split into training and test subsets to fairly assess model generalization. 

---

> ### 📝 3. Data Requirements Report (Summary)
>
> * **Data Source:** The lab will use the widely recognized [Credit Card Fraud Detection dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), which contains real transactions made by European cardholders in September 2013. The dataset consists of 284,807 transactions, with each row representing a single credit card transaction.
> * **Features (Independent Variables):**
>   - `V1`–`V28`: Result of a PCA transformation—these are anonymized, engineered features created by applying Principal Component Analysis (PCA) to the original card transaction variables to protect confidentiality.
>   - `Time`: Seconds elapsed between each transaction and the first transaction in the dataset.
>   - `Amount`: The transaction amount.
> * **Target (Dependent Variable):** `Class`: Binary label indicating transaction status (0 = legitimate, 1 = fraudulent).
> * **Granularity & Privacy:** Each row corresponds to an individual credit card transaction. Raw, identifying information (e.g., card number, merchant, location) is not included. All advanced feature columns except "Time" and "Amount" have been anonymized using PCA to preserve privacy. The dataset is fully anonymized and contains no sensitive or personally identifiable information (PII).

---

## Stage 4: Data Collection


In [12]:
# Necessary imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib


# Configurations
plt.style.use("fivethirtyeight")
sns.set_theme(style="white", palette="colorblind")

### 4.1 Extraction, Transformation, Loading
The data has already been download manually and saved as `../data/raw/creditcard.csv`. For this project, no further ETL process is needed.

### 4.4. Verification

In [13]:
raw_data_path = Path("../data/raw/creditcard.csv")

# Load the final interim data and verify its contents
try:
    df_verified = pd.read_csv(raw_data_path)
    print(f"Verification successful. The following DataFrame is ready for analysis with {df_verified.shape[0]} samples and {df_verified.shape[1]} features:")
    display(df_verified.sample(5))
except Exception as e:
    print(f"An error occurred while verifying the interim data: {e}")

Verification successful. The following DataFrame is ready for analysis with 284807 samples and 31 features:


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
284074,172132.0,-1.346978,0.839384,-0.022076,1.120095,0.138093,-0.734509,0.508762,0.196391,-0.023784,...,0.258456,0.911924,-0.055184,-0.037804,-0.253356,-0.439839,-0.749452,-0.366372,15.83,0
118398,75062.0,-0.913468,-0.525735,0.733373,-0.708056,2.153839,3.703952,-0.800504,1.047173,-1.724929,...,-0.154054,-0.397840,-0.036144,0.956064,0.428332,-0.152400,0.087110,0.101400,69.00,0
22017,31989.0,-1.979918,-5.849500,-1.000495,-0.154425,-2.542383,0.912341,1.196547,-0.035100,1.533918,...,1.196434,0.062554,-1.477042,-0.138265,-0.296497,-0.014869,-0.232431,0.286933,1619.10,0
24509,33265.0,0.400065,-1.591889,0.658872,0.296987,-0.705258,1.762626,-0.554805,0.565419,1.142583,...,0.082119,-0.083075,-0.078848,-0.875800,-0.295179,1.036970,-0.022998,0.055915,331.95,0
134927,81033.0,-0.137780,0.036332,2.382275,-1.031130,-1.214078,-0.557390,-0.459578,0.005494,-0.849744,...,0.409225,1.288120,-0.227049,0.801757,0.015871,-0.110372,0.085950,0.035931,0.00,0



---

> ### 📝 4. Data Collection Report (Summary)
>
> * **Data Extraction:** The credit card transactions dataset was manually downloaded and placed in the project directory as `../data/raw/creditcard.csv`. No automated extraction was required for this lab.
> * **Transformation & Loading:** No additional data transformation or cleaning was performed at this stage. The raw file was loaded directly for analysis, preserving the original structure and content.
> * **Verification:** The raw data file was read successfully, verifying that it contains 284,807 records and 31 features, with a random sample confirming expected format and content for all columns.

---

## Stage 5: Data Understanding

np.int64(0)